# Quantum kernel: fidelity kernel, bandwidth, concentration và shots

Một **quantum kernel** dùng mạch lượng tử làm feature map $$x \mapsto |\varphi(x)\rangle$$ trong không gian $$2^n$$ chiều, rồi
đưa ma trận kernel $$k(x, x') = |\langle \varphi(x) | \varphi(x') \rangle|^2$$ cho một SVM bình thường. Bạn tự dựng feature
map ZZ bằng PennyLane, tính ma trận kernel từ statevector, đo **kernel-target alignment**, rồi xem bốn chuyện mà các bài
quảng cáo QML hay bỏ qua: kernel phải chỉnh **bandwidth** như RBF, kernel **co cụm** (concentration) khi thêm qubit,
chi phí mô phỏng tăng theo $$2^n$$, và trên phần cứng mỗi phần tử kernel chỉ ước lượng được bằng **shots**. Mục 6 lý
thuyết có công thức và bảng kết quả bạn sẽ tái hiện.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) chạy thí nghiệm; test bỏ qua chúng.
- Iris, moons, breast cancer đều có sẵn trong scikit-learn, không cần tải.
- Chấm: `uv run pytest tracks/ai-engineer/modules/22-quantum-ml/exercises/intermediate/02-quantum-kernel -q`

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy
import time
import warnings

import numpy as np
import pennylane as qml

warnings.filterwarnings("ignore", category=UserWarning)

## 1. Feature map ZZ

$$n$$ = số feature = số qubit. Lặp `reps` lần khối sau trên trạng thái $$|0\dots0\rangle$$:

1. Với mỗi wire $$w = 0..n-1$$: `Hadamard(w)` rồi `RZ(x[w])` trên wire $$w$$.
2. Với mỗi cặp kề nhau $$w = 0..n-2$$: `CNOT([w, w+1])`, `RZ(x[w] * x[w+1])` trên wire $$w+1$$, `CNOT([w, w+1])`.

Khối ở bước 2 là $$e^{-i\,x_w x_{w+1} Z_w Z_{w+1}/2}$$ (lên tới một pha toàn cục): nó tạo entanglement phụ thuộc **tích** hai
feature. Đây là bản rút gọn của feature map trong Havlíček et al. (2019).

`feature_states(X, reps=2)`: `X` shape `(B, n)`, trả mảng complex shape `(B, 2**n)`, hàng $$b$$ là statevector
$$|\varphi(x_b)\rangle$$ (wire 0 là bit trái, quy ước PennyLane). Dùng `qml.device("default.qubit", wires=n)`, trả
`qml.state()`, và parameter broadcasting (truyền cả cột `X[:, w]`) để một lần chạy QNode cho cả batch.

In [ ]:
def feature_states(X: np.ndarray, reps: int = 2) -> np.ndarray:
    """Statevector của feature map ZZ cho từng hàng của X: mảng complex shape (B, 2**n), n = X.shape[1]."""
    X = np.atleast_2d(np.asarray(X, dtype=float))
    n = X.shape[1]
    dev = qml.device("default.qubit", wires=n)

    @qml.qnode(dev)
    def circuit(X):
        for _ in range(reps):
            for w in range(n):
                qml.Hadamard(wires=w)
                qml.RZ(X[:, w], wires=w)
            for w in range(n - 1):
                qml.CNOT(wires=[w, w + 1])
                qml.RZ(X[:, w] * X[:, w + 1], wires=w + 1)
                qml.CNOT(wires=[w, w + 1])
        return qml.state()

    return np.asarray(circuit(X)).reshape(len(X), 2**n)

## 2. Ma trận kernel và kernel-target alignment

- `kernel_matrix(XA, XB, reps=2)`: ma trận thực shape `(len(XA), len(XB))`, phần tử $$(i, j)$$ là
  $$|\langle \varphi(a_i) | \varphi(b_j) \rangle|^2$$. Tính statevector **một lần** cho mỗi tập rồi nhân ma trận; không
  chạy một mạch cho mỗi cặp (trên phần cứng thì buộc phải thế, mục 6 lý thuyết).
- `kernel_target_alignment(K, y)`: **centered** kernel-target alignment (Cortes et al., 2012). Ma trận đích
  $$T_{ij} = +1$$ nếu $$y_i = y_j$$, ngược lại $$-1$$. Với $$H = I - \frac{1}{m}\mathbf{1}\mathbf{1}^\top$$ ($$m$$ = số mẫu), đặt
  $$K_c = HKH$$, $$T_c = HTH$$ rồi trả $$A = \frac{\langle K_c, T_c \rangle_F}{\lVert K_c \rVert_F \, \lVert T_c \rVert_F}$$ (float Python).
  Alignment cao nghĩa là kernel coi các điểm cùng lớp là "gần", khác lớp là "xa"; dùng để chọn siêu tham số kernel
  mà không cần train SVM. Bỏ bước centering thì một hằng số cộng vào mọi phần tử của $$K$$ cũng đổi được alignment.
- `sample_kernel(K, shots, rng)`: giả lập ước lượng từng phần tử bằng `shots` lần đo. Trên phần cứng, $$k(a, b)$$ là
  xác suất đo ra toàn số 0 của mạch $$U(b)^\dagger U(a)$$, nên số lần ra toàn 0 theo phân phối nhị thức: trả
  `rng.binomial(shots, K) / shots` (mỗi phần tử độc lập, `K` cắt về $$[0, 1]$$ trước), số ngẫu nhiên chỉ lấy từ `rng`.

In [ ]:
def kernel_matrix(XA: np.ndarray, XB: np.ndarray, reps: int = 2) -> np.ndarray:
    """K[i, j] = |<φ(XA[i]) | φ(XB[j])>|², shape (len(XA), len(XB)), số thực."""
    SA = feature_states(XA, reps)
    SB = feature_states(XB, reps)
    return np.abs(SA.conj() @ SB.T) ** 2                 # <a|b> = Σ conj(a_k) b_k


def kernel_target_alignment(K: np.ndarray, y: np.ndarray) -> float:
    """Centered alignment <HKH, HTH>_F / (||HKH||_F ||HTH||_F), T[i, j] = +1 nếu y[i] == y[j], ngược lại −1."""
    y = np.asarray(y)
    T = np.where(y[:, None] == y[None, :], 1.0, -1.0)
    m = len(y)
    H = np.eye(m) - np.ones((m, m)) / m                  # centering: trừ trung bình hàng và cột
    Kc, Tc = H @ K @ H, H @ T @ H
    return float(np.sum(Kc * Tc) / (np.linalg.norm(Kc) * np.linalg.norm(Tc)))


def sample_kernel(K: np.ndarray, shots: int, rng: np.random.Generator) -> np.ndarray:
    """Ước lượng từng phần tử của K bằng shots lần đo: rng.binomial(shots, clip(K, 0, 1)) / shots."""
    return rng.binomial(shots, np.clip(K, 0.0, 1.0)) / shots

In [ ]:
# explore: kiểm nhanh tính chất của kernel trên 6 điểm ngẫu nhiên 3 feature
Xq = np.random.default_rng(0).uniform(-1, 1, size=(6, 3))
Kq = kernel_matrix(Xq, Xq)
print("đường chéo:", np.round(np.diag(Kq), 12))
print("đối xứng:", np.allclose(Kq, Kq.T), "| trị riêng nhỏ nhất:", np.linalg.eigvalsh(Kq).min())

## 3. Bandwidth: kernel lượng tử cũng phải chỉnh như RBF

Input được chuẩn hoá (`StandardScaler` fit trên fold train) rồi **nhân với `scale`** trước khi vào mạch. `scale` đóng
vai trò như $$\gamma$$ của RBF: nhỏ thì mọi điểm "giống nhau" (kernel gần toàn 1), lớn thì mọi điểm "khác nhau" (kernel
gần ma trận đơn vị, SVM thuộc lòng train). Cell dưới chạy 5-fold CV (stratified, `random_state=0`) với SVM
`kernel="precomputed"`, `C=1`, so với SVM RBF và logistic regression cùng 5 fold. Cả cell khoảng 5 giây.

In [ ]:
# explore: Iris (4 feature = 4 qubit, 3 lớp) và moons (2 qubit): quét scale, so với RBF và logistic (bảng mục 6)
from sklearn.datasets import load_breast_cancer, load_iris, make_moons
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

CV = StratifiedKFold(5, shuffle=True, random_state=0)


def cv_quantum_kernel(X, y, scale, prep=StandardScaler, shots=None, seed=0):
    """5-fold CV của SVM với kernel ZZ. Trả dict: train/test accuracy, alignment, thống kê phần tử ngoài đường chéo."""
    rng = np.random.default_rng(seed)
    out = {"train": [], "test": [], "align": [], "off_mean": [], "off_std": []}
    for tr, te in CV.split(X, y):
        p = prep().fit(X[tr])
        A, B = p.transform(X[tr]) * scale, p.transform(X[te]) * scale
        K_tr, K_te = kernel_matrix(A, A), kernel_matrix(B, A)
        if shots:
            K_tr = np.triu(sample_kernel(K_tr, shots, rng), 1)
            K_tr = K_tr + K_tr.T + np.eye(len(K_tr))         # chỉ đo cặp i < j, đường chéo luôn bằng 1
            K_te = sample_kernel(K_te, shots, rng)
        clf = SVC(kernel="precomputed", C=1.0).fit(K_tr, y[tr])
        off = K_tr[~np.eye(len(K_tr), dtype=bool)]
        out["train"].append(clf.score(K_tr, y[tr]))
        out["test"].append(clf.score(K_te, y[te]))
        out["align"].append(kernel_target_alignment(K_tr, y[tr]))
        out["off_mean"].append(off.mean())
        out["off_std"].append(off.std())
    return {k: float(np.mean(v)) for k, v in out.items()} | {"test_std": float(np.std(out["test"]))}


X_iris, y_iris = load_iris(return_X_y=True)
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=0)
for name, X, y in [("Iris", X_iris, y_iris), ("moons", X_moons, y_moons)]:
    print(name)
    for scale in (0.1, 0.3, 0.5, 1.0, 2.0, 4.0):
        r = cv_quantum_kernel(X, y, scale)
        print(f"  scale {scale:3.1f}: train {r['train']:.3f} | test {r['test']:.3f} ± {r['test_std']:.3f}"
              f" | alignment {r['align']:.3f} | ngoài đường chéo: mean {r['off_mean']:.3f}, std {r['off_std']:.3f}")
    for cname, clf in [("SVM RBF", make_pipeline(StandardScaler(), SVC())),
                       ("Logistic", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))]:
        s = cross_val_score(clf, X, y, cv=CV)
        print(f"  {cname:9s}: test {s.mean():.3f} ± {s.std():.3f}")

## 4. Concentration: thêm qubit thì kernel co về 0

Breast cancer (569 mẫu, 30 feature, có sẵn trong scikit-learn), giảm xuống $$n$$ thành phần PCA rồi chuẩn hoá lại, dùng
$$n$$ qubit với $$n = 2, 4, \dots, 10$$. Với `scale = 1.0`, khi $$n$$ tăng các phần tử ngoài đường chéo co dần về 0: hai điểm
bất kỳ gần như trực giao trong không gian $$2^n$$ chiều, kernel tiến về ma trận đơn vị, SVM thuộc lòng train mà không
tổng quát hoá. Giảm `scale` kéo kernel về lại vùng dùng được. Để ý cả cột thời gian: mỗi 2 qubit thêm làm statevector
dài gấp 4. Cả cell khoảng 25 giây.

In [ ]:
# explore: breast cancer, n thành phần PCA = n qubit; scale 1.0 và 0.3 (bảng concentration mục 6)
X_bc, y_bc = load_breast_cancer(return_X_y=True)
for n in (2, 4, 6, 8, 10):
    prep = lambda n=n: make_pipeline(StandardScaler(), PCA(n, random_state=0), StandardScaler())
    rbf = cross_val_score(make_pipeline(prep(), SVC()), X_bc, y_bc, cv=CV).mean()
    for scale in (1.0, 0.3):
        start = time.perf_counter()
        r = cv_quantum_kernel(X_bc, y_bc, scale, prep=prep)
        print(f"n={n:2d} scale {scale}: train {r['train']:.3f} | test {r['test']:.3f} | ngoài đường chéo mean"
              f" {r['off_mean']:.3f} std {r['off_std']:.3f} | SVM RBF {rbf:.3f} | {time.perf_counter() - start:.1f} s")

## 5. Shots: trên phần cứng mỗi phần tử kernel là một ước lượng

Mô phỏng statevector cho kernel chính xác. Phần cứng thì cho mỗi cặp $$(a, b)$$ một mạch, đo `shots` lần, đếm tỉ lệ ra
toàn 0. Cell dưới thay kernel đúng bằng `sample_kernel` trên Iris (`scale = 0.5`) và đếm số mạch phải chạy.

In [ ]:
# explore: Iris scale 0.5 với kernel ước lượng từ shots, và số mạch cần chạy cho một fold
for shots in (10, 100, 1000, None):
    r = cv_quantum_kernel(X_iris, y_iris, 0.5, shots=shots, seed=0)
    label = f"{shots} shots" if shots else "chính xác"
    print(f"{label:11s}: test {r['test']:.3f} ± {r['test_std']:.3f} | alignment {r['align']:.3f}")
n_tr, n_te = 120, 30
n_circuits = n_tr * (n_tr - 1) // 2 + n_te * n_tr
print(f"một fold Iris: {n_circuits} mạch khác nhau; với 1000 shots mỗi mạch là {n_circuits * 1000:,} lần đo")

## 6. Đọc kết quả

Trước khi làm trắc nghiệm trong app, tự trả lời:

1. Trên Iris, scale nào cho test accuracy tốt nhất, và nó so với SVM RBF, logistic thế nào? Trên moons thì sao?
2. Alignment có chọn được scale tốt mà không cần train SVM không? Nó sai ở đâu?
3. Ở bảng concentration, vì sao train accuracy **tăng** còn test accuracy **giảm** khi thêm qubit với scale 1.0?
4. Với 100 shots, kernel lệch bao nhiêu so với chính xác (độ lệch chuẩn nhị thức), và accuracy đổi bao nhiêu?